In [22]:
import pandas as pd


df = pd.read_parquet('cleaned_hospital_data_final.parquet')

In [23]:
df.duplicated().sum().sum()

0

In [24]:
df.isna().sum().sum()

0

In [ ]:
for col in df.select_dtypes(include=['object', 'category']).columns[:10]:
    if (df[col] == ' ').any() or (df[col] == '').any():
        print(f"Warning: Blank spaces found in {col}")

In [ ]:
for col in ['Total Charges', 'Total Costs']:
    if df[col].dtype == 'object':
        df[col] = (
            df[col]
            .astype(str)
            .str.replace('$', '', regex=False)
            .str.replace(',', '', regex=False)
            .str.strip()
        )
        df[col] = pd.to_numeric(df[col], errors='coerce')

print("Zero or Negative Charges:", (df['Total Charges'] <= 0).sum())
print("Zero or Negative Costs:", (df['Total Costs'] <= 0).sum())

Zero or Negative Charges: 1
Zero or Negative Costs: 1


In [ ]:
print("Zero or Negative Charges:", (df['Total Charges'] <= 0).sum())
print("Zero or Negative Costs:", (df['Total Costs'] <= 0).sum())

Zero or Negative Charges: 1
Zero or Negative Costs: 1


In [ ]:
df = df[(df['Total Charges'] > 0) & (df['Total Costs'] > 0)]

print("Done! Remaining rows:", len(df))

Done! Remaining rows: 4184883


In [29]:
print("Data type of Length of Stay:", df['Length of Stay'].dtype)
if df['Length of Stay'].dtype == 'object':
    print("Unique non-numeric values:", df[~df['Length of Stay'].astype(str).str.isnumeric()]['Length of Stay'].unique())

Data type of Length of Stay: object
Unique non-numeric values: ['120 +']


In [ ]:
df['Length of Stay'] = df['Length of Stay'].replace('120 +', '120')

df['Length of Stay'] = pd.to_numeric(df['Length of Stay'], errors='coerce').astype('int32')

print("New Data Type:", df['Length of Stay'].dtype)
print("Max Stay:", df['Length of Stay'].max())
print("Min Stay:", df['Length of Stay'].min())

New Data Type: int32
Max Stay: 120
Min Stay: 1


In [31]:
df['Discharge Year'].value_counts()

Discharge Year
2023    2103774
2022    2081109
Name: count, dtype: int64

In [ ]:
cols_to_check = ['Gender', 'Race', 'Ethnicity', 'Age Group', 'Type of Admission']
for col in cols_to_check:
    print(f"=== {col} ===")
    print(df[col].unique())
    print()

=== Gender ===
['M' 'F' 'U']

=== Race ===
['Other Race' 'White' 'Black/African American' 'Multi-racial']

=== Ethnicity ===
['Spanish/Hispanic' 'Unknown' 'Not Span/Hispanic' 'Multi-ethnic']

=== Age Group ===
['18 to 29' '50 to 69' '70 or Older' '30 to 49' '0 to 17']

=== Type of Admission ===
['Emergency' 'Elective' 'Urgent' 'Not Available' 'Newborn' 'Trauma']



In [33]:
print("Emergency Indicator values:\n", df['Emergency Department Indicator'].value_counts())

Emergency Indicator values:
 Emergency Department Indicator
Y    2665622
N    1519261
Name: count, dtype: int64


In [ ]:
abnormal_costs = (df['Total Costs'] > df['Total Charges']).sum()
print("Cases where Costs > Charges:", abnormal_costs)

Cases where Costs > Charges: 52455


In [35]:
df['Emergency Department Indicator'] = (df['Emergency Department Indicator'] == 'Y').astype('int8')

In [ ]:

extreme_cases = df[df['Total Costs'] > (df['Total Charges'] * 5)]
print("Extreme anomalies count:", len(extreme_cases))

Extreme anomalies count: 130


In [ ]:
df = df[~(df['Total Costs'] > (df['Total Charges'] * 5))]

print("Removed 130 extreme anomalies. Remaining rows:", len(df))

Removed 130 extreme anomalies. Remaining rows: 4184753


In [38]:
df.info(memory_usage='deep')

<class 'pandas.core.frame.DataFrame'>
Index: 4184753 entries, 0 to 4184883
Data columns (total 32 columns):
 #   Column                               Dtype  
---  ------                               -----  
 0   Hospital Service Area                object 
 1   Hospital County                      object 
 2   Operating Certificate Number         float64
 3   Permanent Facility Id                float64
 4   Facility Name                        object 
 5   Age Group                            object 
 6   Zip Code - 3 digits                  object 
 7   Gender                               object 
 8   Race                                 object 
 9   Ethnicity                            object 
 10  Length of Stay                       int32  
 11  Type of Admission                    object 
 12  Patient Disposition                  object 
 13  Discharge Year                       int64  
 14  CCSR Diagnosis Code                  object 
 15  CCSR Diagnosis Description           

In [ ]:
categorical_cols = [
    'Hospital Service Area', 'Hospital County', 'Age Group', 'Zip Code - 3 digits',
    'Gender', 'Race', 'Ethnicity', 'Type of Admission', 'Patient Disposition',
    'APR Severity of Illness Description', 'APR Risk of Mortality',
    'APR Medical Surgical Description', 'Payment Typology 1'
]

for col in categorical_cols:
    df[col] = df[col].astype('category')

df['Discharge Year'] = df['Discharge Year'].astype('int16')
df['APR DRG Code'] = df['APR DRG Code'].astype('int16')
df['APR MDC Code'] = df['APR MDC Code'].astype('int8')
df['APR Severity of Illness Code'] = df['APR Severity of Illness Code'].astype('int8')
df['Is_Neonate'] = df['Is_Neonate'].astype('int8')

# 3. تحويل المعرفات من float إلى int32 (لتوفير المساحة)
df['Operating Certificate Number'] = df['Operating Certificate Number'].astype('int32')
df['Permanent Facility Id'] = df['Permanent Facility Id'].astype('int32')

df.info(memory_usage='deep')

<class 'pandas.core.frame.DataFrame'>
Index: 4184753 entries, 0 to 4184883
Data columns (total 32 columns):
 #   Column                               Dtype   
---  ------                               -----   
 0   Hospital Service Area                category
 1   Hospital County                      category
 2   Operating Certificate Number         int32   
 3   Permanent Facility Id                int32   
 4   Facility Name                        object  
 5   Age Group                            category
 6   Zip Code - 3 digits                  category
 7   Gender                               category
 8   Race                                 category
 9   Ethnicity                            category
 10  Length of Stay                       int32   
 11  Type of Admission                    category
 12  Patient Disposition                  category
 13  Discharge Year                       int16   
 14  CCSR Diagnosis Code                  object  
 15  CCSR Diagnosis Descr

In [40]:
df['Length of Stay'] = df['Length of Stay'].replace('120 +', '120').astype('int32')

In [ ]:
df.to_parquet('cleaned_hospital_data_final.parquet', index=False)
print("Saved successfully! Size on disk")

Saved successfully! Size on disk


In [ ]:
df.to_csv('cleaned_hospital_data.csv', index=False)
print("CSV file created successfully!")

CSV file created successfully!
